In [11]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from datetime import datetime, timezone
from urllib.parse import urljoin
from typing import List, Dict, Any, Optional
import xml.etree.ElementTree as ET
import re
from html import unescape
import logging
from dateutil import parser as date_parser
import sys
import os
import time

# Add the functions directory to the path
sys.path.append('/Users/armanpani/development/odiya-genai-backend/functions')

logger = logging.getLogger(__name__)

from database.crud_operations import batch_check_duplicates
# from scraping.summarize_article import summarize_articles_batch
from database.postsql_db_connection import test_database_connection


In [12]:

# Configuration for multiple Odisha news websites with RSS feeds
NEWS_WEBSITES = {
    "odishatv": {
        "base_url": "https://odishatv.in",
        "rss_url": "https://odishatv.in/rss",
        "source_name": "OdishaTV",
        "url_patterns": ["https://odishatv.in/odisha/"]
    },
    "odishabytes": {
        "base_url": "https://odishabytes.com",
        "rss_url": "https://odishabytes.com/category/odisha/rss",
        "source_name": "OdishaBytes",
        "url_patterns": ["https://odishabytes.com/"]
    },
    "sambadenglish": {
        "base_url": "https://sambadenglish.com",
        "rss_url": "https://sambadenglish.com/rss",
        "source_name": "Sambad English",
        "url_patterns": ["/latest-news/", "/news-from-around-the-state/"]
    },
    "orissapost": {
        "base_url": "https://www.orissapost.com",
        "rss_url": "https://www.orissapost.com/state-news/rss",
        "source_name": "Orissa Post",
        "url_patterns": ["https://www.orissapost.com"]
    },
}


In [ ]:

def create_session_with_retries() -> requests.Session:
    """Create a requests session with retry strategy"""
    session = requests.Session()
    
    retry_strategy = Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
    )
    
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    
    return session

def matches_url_pattern(url: str, patterns: List[str], base_url: str) -> bool:
    """Check if URL matches any of the specified patterns, handling relative URLs"""
    for pattern in patterns:
        # If pattern is relative, make it absolute
        if not pattern.startswith('http'):
            pattern = urljoin(base_url, pattern)
        
        if pattern in url:
            return True
    return False


def clean_html_content(html_content: str) -> str:
    """Clean HTML content and extract plain text"""
    if not html_content:
        return ""
    
    # Remove HTML tags
    clean_text = re.sub(r'<[^>]+>', '', html_content)
    
    # Decode HTML entities
    clean_text = unescape(clean_text)
    
    # Clean up whitespace
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()
    
    return clean_text


def parse_rss_date(date_str: str) -> str:
    """Parse RSS date string using dateutil for robust parsing"""
    try:
        # Use dateutil.parser for robust date parsing
        dt = date_parser.parse(date_str)
        
        # Ensure timezone awareness
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        
        return dt.strftime("%Y-%m-%d %H:%M:%S")
        
    except Exception as e:
        logger.warning(f"Could not parse date '{date_str}': {e}")
        return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")


def extract_image_url(item: ET.Element, base_url: str) -> str:
    """Extract image URL from RSS item, checking multiple sources with enhanced debugging"""
    image_url = ""
    source_found = ""
    
    logger.debug(f"🔍 Extracting image URL from RSS item")
    
    # Check enclosure tag
    enclosure = item.find('enclosure')
    if enclosure is not None:
        enclosure_type = enclosure.get('type', '')
        enclosure_url = enclosure.get('url', '')
        logger.debug(f"   Enclosure found: type='{enclosure_type}', url='{enclosure_url}'")
        if enclosure_type.startswith('image'):
            image_url = enclosure_url
            source_found = "enclosure"
    
    # Check media:content tag
    if not image_url:
        media_content = item.find('.//{http://search.yahoo.com/mrss/}content')
        if media_content is not None:
            content_type = media_content.get('type', '')
            content_url = media_content.get('url', '')
            logger.debug(f"   Media content found: type='{content_type}', url='{content_url}'")
            if content_type.startswith('image'):
                image_url = content_url
                source_found = "media:content"
    
    # Check media:thumbnail tag
    if not image_url:
        media_thumbnail = item.find('.//{http://search.yahoo.com/mrss/}thumbnail')
        if media_thumbnail is not None:
            thumbnail_url = media_thumbnail.get('url', '')
            logger.debug(f"   Media thumbnail found: url='{thumbnail_url}'")
            image_url = thumbnail_url
            source_found = "media:thumbnail"
    
    # Check for image tags in description/content
    if not image_url:
        description = item.find('description')
        if description is not None and description.text:
            # Look for img tags in description
            import re
            img_pattern = r'<img[^>]+src=["\']([^"\']+)["\'][^>]*>'
            img_matches = re.findall(img_pattern, description.text, re.IGNORECASE)
            if img_matches:
                image_url = img_matches[0]
                source_found = "description img tag"
                logger.debug(f"   Image found in description: {image_url}")
    
    # Check for content:encoded with images
    if not image_url:
        content_encoded = item.find('.//{http://purl.org/rss/1.0/modules/content/}encoded')
        if content_encoded is not None and content_encoded.text:
            import re
            img_pattern = r'<img[^>]+src=["\']([^"\']+)["\'][^>]*>'
            img_matches = re.findall(img_pattern, content_encoded.text, re.IGNORECASE)
            if img_matches:
                image_url = img_matches[0]
                source_found = "content:encoded img tag"
                logger.debug(f"   Image found in content:encoded: {image_url}")
    
    # Check for WordPress featured image
    if not image_url:
        wp_thumbnail = item.find('.//{http://wordpress.org/export/1.2/}post_thumbnail')
        if wp_thumbnail is not None and wp_thumbnail.text:
            image_url = wp_thumbnail.text
            source_found = "wp:post_thumbnail"
            logger.debug(f"   WordPress thumbnail found: {image_url}")
    
    # Make relative URLs absolute
    if image_url and not image_url.startswith('http'):
        original_url = image_url
        image_url = urljoin(base_url, image_url)
        logger.debug(f"   Converted relative URL: {original_url} -> {image_url}")
    
    if image_url:
        logger.debug(f"✅ Image URL found via {source_found}: {image_url}")
    else:
        logger.debug(f"❌ No image URL found in any source")
        # Log all available tags for debugging
        all_tags = [child.tag for child in item]
        logger.debug(f"   Available tags: {all_tags}")
    
    return image_url


def debug_image_extraction(source_key: str, max_articles: int = 2):
    """Debug image extraction for a specific source"""
    print(f"\n🔍 DEBUGGING IMAGE EXTRACTION FOR {source_key.upper()}")
    print("=" * 60)
    
    if source_key not in NEWS_WEBSITES:
        print(f"❌ Source '{source_key}' not found")
        return
    
    website_config = NEWS_WEBSITES[source_key]
    print(f"📡 RSS URL: {website_config['rss_url']}")
    
    try:
        # Fetch RSS feed
        session = create_session_with_retries()
        response = session.get(website_config["rss_url"], timeout=15)
        
        if response.status_code != 200:
            print(f"❌ Failed to fetch RSS: {response.status_code}")
            return
        
        # Parse XML
        root = ET.fromstring(response.content)
        items = root.findall('.//item')[:max_articles]
        
        print(f"📰 Found {len(items)} RSS items to analyze")
        
        for i, item in enumerate(items, 1):
            print(f"\n📰 ANALYZING ITEM #{i}")
            print("-" * 40)
            
            # Get basic info
            title = item.find('title')
            link = item.find('link')
            description = item.find('description')
            
            print(f"Title: {title.text if title is not None else 'N/A'}")
            print(f"Link: {link.text if link is not None else 'N/A'}")
            
            # Check all possible image sources
            print(f"\n🔍 CHECKING IMAGE SOURCES:")
            
            # 1. Enclosure
            enclosure = item.find('enclosure')
            if enclosure is not None:
                print(f"   Enclosure: type='{enclosure.get('type', '')}', url='{enclosure.get('url', '')}'")
            else:
                print(f"   Enclosure: Not found")
            
            # 2. Media content
            media_content = item.find('.//{http://search.yahoo.com/mrss/}content')
            if media_content is not None:
                print(f"   Media content: type='{media_content.get('type', '')}', url='{media_content.get('url', '')}'")
            else:
                print(f"   Media content: Not found")
            
            # 3. Media thumbnail
            media_thumbnail = item.find('.//{http://search.yahoo.com/mrss/}thumbnail')
            if media_thumbnail is not None:
                print(f"   Media thumbnail: url='{media_thumbnail.get('url', '')}'")
            else:
                print(f"   Media thumbnail: Not found")
            
            # 4. Description with images
            if description is not None and description.text:
                import re
                img_pattern = r'<img[^>]+src=["\']([^"\']+)["\'][^>]*>'
                img_matches = re.findall(img_pattern, description.text, re.IGNORECASE)
                if img_matches:
                    print(f"   Description images: {img_matches}")
                else:
                    print(f"   Description images: Not found")
            
            # 5. All available tags
            all_tags = [child.tag for child in item]
            print(f"   All available tags: {all_tags}")
            
            # Test our enhanced extraction
            image_url = extract_image_url(item, website_config['base_url'])
            print(f"\n🎯 FINAL RESULT: {image_url if image_url else 'No image found'}")
            
    except Exception as e:
        print(f"❌ Error during debugging: {e}")


def extract_author(item: ET.Element) -> List[str]:
    """Extract author information from RSS item"""
    authors = []
    
    # Check for dc:creator
    creator = item.find('.//{http://purl.org/dc/elements/1.1/}creator')
    if creator is not None and creator.text:
        authors.append(creator.text.strip())
    
    # Check for author tag
    author = item.find('author')
    if author is not None and author.text:
        authors.append(author.text.strip())
    
    return authors


def parse_rss_items(root: ET.Element, website_config: Dict[str, Any], max_articles: Optional[int] = None) -> List[Dict[str, Any]]:
    """Parse RSS 2.0 items"""
    articles = []
    base_url = website_config["base_url"]
    
    items = root.findall('.//item')
    if max_articles:
        items = items[:max_articles]
    
    for item in items:
        try:
            # Extract basic information
            link_elem = item.find('link')
            title_elem = item.find('title')
            description_elem = item.find('description')
            pub_date_elem = item.find('pubDate')
            
            if link_elem is None or link_elem.text is None:
                continue
                
            url = link_elem.text.strip()
            
            # Check if URL matches patterns
            if not matches_url_pattern(url, website_config['url_patterns'], base_url):
                continue
            
            # Extract title
            title = ""
            if title_elem is not None and title_elem.text:
                title = clean_html_content(title_elem.text)
            
            # Extract content from description
            content = ""
            if description_elem is not None and description_elem.text:
                content = clean_html_content(description_elem.text)
            
            # Skip if no title or content
            if not title or not content:
                logger.debug(f"Skipping article with missing content: {url}")
                continue
            
            # Extract publish date
            publish_date = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
            if pub_date_elem is not None and pub_date_elem.text:
                publish_date = parse_rss_date(pub_date_elem.text)
            
            # Extract image URL
            image_url = extract_image_url(item, base_url)
            
            # Extract authors
            authors = extract_author(item)
            
            articles.append({
                "original_title": title,
                "original_content": content,
                "image_url": image_url,
                "url": url,
                "source_name": website_config["source_name"],
                "authors": authors,
                "publish_date": publish_date
            })
            
            logger.info(f"Extracted article from {website_config['source_name']}: {title[:50]}...")
            
        except Exception as e:
            logger.error(f"Error processing RSS item: {e}")
            continue
    
    return articles


def parse_atom_entries(root: ET.Element, website_config: Dict[str, Any], max_articles: Optional[int] = None) -> List[Dict[str, Any]]:
    """Parse Atom feed entries"""
    articles = []
    base_url = website_config["base_url"]
    
    entries = root.findall('.//{http://www.w3.org/2005/Atom}entry')
    if max_articles:
        entries = entries[:max_articles]
    
    for entry in entries:
        try:
            link_elem = entry.find('{http://www.w3.org/2005/Atom}link')
            title_elem = entry.find('{http://www.w3.org/2005/Atom}title')
            summary_elem = entry.find('{http://www.w3.org/2005/Atom}summary')
            updated_elem = entry.find('{http://www.w3.org/2005/Atom}updated')
            
            if link_elem is None:
                continue
                
            href = link_elem.get('href')
            if not href:
                continue
            
            # Make relative URLs absolute
            if not href.startswith('http'):
                href = urljoin(base_url, href)
            
            # Check if URL matches patterns
            if not matches_url_pattern(href, website_config['url_patterns'], base_url):
                continue
            
            # Extract title
            title = ""
            if title_elem is not None and title_elem.text:
                title = clean_html_content(title_elem.text)
            
            # Extract content
            content = ""
            if summary_elem is not None and summary_elem.text:
                content = clean_html_content(summary_elem.text)
            
            # Skip if no title or content
            if not title or not content:
                logger.debug(f"Skipping article with missing content: {href}")
                continue
            
            # Extract publish date
            publish_date = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
            if updated_elem is not None and updated_elem.text:
                publish_date = parse_rss_date(updated_elem.text)
            
            # Extract image URL (for Atom feeds)
            image_url = extract_image_url(entry, base_url)
            
            # Extract authors
            authors = []
            author_elem = entry.find('{http://www.w3.org/2005/Atom}author')
            if author_elem is not None:
                name_elem = author_elem.find('{http://www.w3.org/2005/Atom}name')
                if name_elem is not None and name_elem.text:
                    authors.append(name_elem.text.strip())
            
            articles.append({
                "original_title": title,
                "original_content": content,
                "image_url": image_url,
                "url": href,
                "source_name": website_config["source_name"],
                "authors": authors,
                "publish_date": publish_date
            })
            
            logger.info(f"Extracted article from {website_config['source_name']}: {title[:50]}...")
            
        except Exception as e:
            logger.error(f"Error processing Atom entry: {e}")
            continue
    
    return articles


def extract_articles_from_rss(website_config: Dict[str, Any], max_articles: Optional[int] = None) -> List[Dict[str, Any]]:
    """Extract article data directly from RSS feed"""
    try:
        logger.info(f"Fetching RSS feed from {website_config['source_name']}: {website_config['rss_url']}")
        
        session = create_session_with_retries()
        response = session.get(
            website_config["rss_url"], 
            headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}, 
            timeout=15
        )
        
        if response.status_code != 200:
            logger.error(f"Failed to fetch RSS feed from {website_config['source_name']}: {response.status_code}")
            return []
        
        # Parse XML RSS feed
        try:
            root = ET.fromstring(response.content)
        except ET.ParseError as e:
            logger.error(f"XML parsing error for {website_config['source_name']}: {e}")
            return []
        
        articles = []
        
        # Try RSS 2.0 format first
        rss_items = root.findall('.//item')
        if rss_items:
            logger.debug(f"Parsing RSS 2.0 format with {len(rss_items)} items")
            articles = parse_rss_items(root, website_config, max_articles)
        else:
            # Try Atom format
            atom_entries = root.findall('.//{http://www.w3.org/2005/Atom}entry')
            if atom_entries:
                logger.debug(f"Parsing Atom format with {len(atom_entries)} entries")
                articles = parse_atom_entries(root, website_config, max_articles)
        
        logger.info(f"Extracted {len(articles)} articles from {website_config['source_name']}")
        return articles
        
    except Exception as e:
        logger.error(f"Error extracting articles from {website_config['source_name']} RSS: {e}")
        return []


def display_articles_info(articles: List[Dict[str, Any]], title: str = "Articles Information"):
    """Display comprehensive information about scraped articles"""
    print(f"\n{'='*80}")
    print(f"📰 {title}")
    print(f"{'='*80}")
    
    if not articles:
        print("❌ No articles to display")
        return
    
    print(f"📊 Total Articles: {len(articles)}")
    print(f"⏰ Display Time: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
    print(f"{'='*80}")
    
    # Group articles by source
    articles_by_source = {}
    for article in articles:
        source = article['source_name']
        if source not in articles_by_source:
            articles_by_source[source] = []
        articles_by_source[source].append(article)
    
    # Display articles by source
    for source_name, source_articles in articles_by_source.items():
        print(f"\n🔍 SOURCE: {source_name}")
        print(f"📊 Articles from this source: {len(source_articles)}")
        print(f"{'-'*60}")
        
        for i, article in enumerate(source_articles, 1):
            print(f"\n📰 Article #{i}")
            print(f"   Title: {article['original_title']}")
            print(f"   URL: {article['url']}")
            print(f"   Published: {article['publish_date']}")
            print(f"   Authors: {', '.join(article['authors']) if article['authors'] else 'Not specified'}")
            print(f"   Image: {'Yes' if article['image_url'] else 'No'}")
            if article['image_url']:
                print(f"   Image URL: {article['image_url']}")
            print(f"   Content Length: {len(article['original_content'])} characters")
            print(f"   Content Preview: {article['original_content'][:200]}{'...' if len(article['original_content']) > 200 else ''}")
            print(f"   {'-'*40}")
    
    # Summary statistics
    print(f"\n📈 SUMMARY STATISTICS")
    print(f"{'='*60}")
    
    # Content length statistics
    content_lengths = [len(article['original_content']) for article in articles]
    print(f"📏 Content Length Statistics:")
    print(f"   Average: {sum(content_lengths) / len(content_lengths):.0f} characters")
    print(f"   Shortest: {min(content_lengths)} characters")
    print(f"   Longest: {max(content_lengths)} characters")
    
    # Image statistics
    articles_with_images = sum(1 for article in articles if article['image_url'])
    print(f"\n🖼️  Image Statistics:")
    print(f"   Articles with images: {articles_with_images}/{len(articles)} ({articles_with_images/len(articles)*100:.1f}%)")
    
    # Author statistics
    articles_with_authors = sum(1 for article in articles if article['authors'])
    print(f"\n👤 Author Statistics:")
    print(f"   Articles with authors: {articles_with_authors}/{len(articles)} ({articles_with_authors/len(articles)*100:.1f}%)")
    
    # Source distribution
    print(f"\n📊 Source Distribution:")
    for source_name, source_articles in articles_by_source.items():
        print(f"   {source_name}: {len(source_articles)} articles ({len(source_articles)/len(articles)*100:.1f}%)")
    
    print(f"\n{'='*80}")


def display_articles_table(articles: List[Dict[str, Any]]):
    """Display articles in a compact table format"""
    if not articles:
        print("❌ No articles to display")
        return
    
    print(f"\n📋 ARTICLES TABLE VIEW")
    print(f"{'='*120}")
    print(f"{'#':<3} {'Source':<15} {'Title':<50} {'Authors':<20} {'Image':<6} {'Length':<8}")
    print(f"{'-'*120}")
    
    for i, article in enumerate(articles, 1):
        source = article['source_name'][:14]
        title = article['original_title'][:49] + ('...' if len(article['original_title']) > 49 else '')
        authors = ', '.join(article['authors'][:2]) if article['authors'] else 'N/A'
        if len(authors) > 19:
            authors = authors[:16] + '...'
        image = 'Yes' if article['image_url'] else 'No'
        length = len(article['original_content'])
        
        print(f"{i:<3} {source:<15} {title:<50} {authors:<20} {image:<6} {length:<8}")
    
    print(f"{'-'*120}")
    print(f"Total: {len(articles)} articles")


def scrape_and_process_articles(max_articles_per_source: Optional[int] = None):
    """Main function to scrape, process, and save articles from all Odisha news websites using RSS feeds"""
    logger.info(f"🚀 Starting RSS-based Odisha news scraping at: {datetime.now(timezone.utc).isoformat()}")
    
    # Test connection first
    logger.info(f"🔍 Testing database connection...")
    if not test_database_connection():
        logger.error("❌ Cannot proceed without database connection")
        return 0
    logger.info(f"✅ Database connection successful")
    
    try:
        all_articles = []
        scraping_start_time = time.time()
        
        # Extract articles from all RSS feeds
        logger.info(f"📡 Starting RSS feed extraction from {len(NEWS_WEBSITES)} sources")
        for website_key, website_config in NEWS_WEBSITES.items():
            logger.info(f"🔄 Processing {website_config['source_name']} RSS...")
            articles = extract_articles_from_rss(website_config, max_articles_per_source)
            all_articles.extend(articles)
            logger.info(f"✅ {website_config['source_name']}: {len(articles)} articles extracted")
        
        scraping_time = time.time() - scraping_start_time
        logger.info(f"⏱️  Total scraping time: {scraping_time:.2f} seconds")
        logger.info(f"📊 Total articles extracted from all RSS feeds: {len(all_articles)}")
        
        if not all_articles:
            logger.warning("⚠️  No articles found to process")
            return 0
        
        # Batch check for duplicates
        logger.info(f"🔍 Checking for duplicate articles...")
        duplicate_start_time = time.time()
        source_urls = [article["url"] for article in all_articles]
        duplicates = batch_check_duplicates(source_urls)
        duplicate_time = time.time() - duplicate_start_time
        
        logger.info(f"⏱️  Duplicate check completed in {duplicate_time:.2f} seconds")
        logger.info(f"🔍 Found {len(duplicates)} duplicate URLs")
        
        # Filter out duplicates
        unique_articles = [article for article in all_articles if article["url"] not in duplicates]
        logger.info(f"✅ Filtered out {len(duplicates)} duplicate articles, {len(unique_articles)} unique articles remaining")
        
        if not unique_articles:
            logger.warning("⚠️  No new articles to process after duplicate filtering")
            return 0

        # Display comprehensive article information
        logger.info(f"📋 Displaying article information...")
        display_articles_info(unique_articles, "SCRAPED ARTICLES INFORMATION")
        
        # Also display in table format for quick overview
        display_articles_table(unique_articles)
        
        # For testing purposes, we'll just return the count of unique articles
        # without summarization or database saving
        total_time = time.time() - scraping_start_time
        logger.info(f"🎉 RSS-based scraping completed at: {datetime.now(timezone.utc).isoformat()}")
        logger.info(f"⏱️  Total processing time: {total_time:.2f} seconds")
        logger.info(f"📊 Found {len(unique_articles)} unique articles (summarization and saving disabled for testing)")
        
        return len(unique_articles)
        
    except Exception as e:
        logger.error(f"💥 Error in RSS-based scraping function: {e}")
        logger.error(f"🔍 Error type: {type(e).__name__}")
        return 0

In [14]:
# Run the scraping process with comprehensive logging and article display
print("🚀 Starting News Scraping Process...")
print("=" * 60)

# Set logging level to INFO to see all the detailed logs
logging.getLogger().setLevel(logging.INFO)

# Run the scraping process
result = scrape_and_process_articles(max_articles_per_source=5)

print(f"\n🎯 SCRAPING COMPLETED!")
print(f"📊 Total unique articles found: {result}")
print("=" * 60)

🚀 Starting News Scraping Process...
✅ Database connected: PostgreSQL 15.14 on x86_64-pc-linux-gnu, compiled by Debian clang version 12.0.1, 64-bit

📰 SCRAPED ARTICLES INFORMATION
📊 Total Articles: 13
⏰ Display Time: 2025-09-29 19:18:29 UTC

🔍 SOURCE: OdishaTV
📊 Articles from this source: 4
------------------------------------------------------------

📰 Article #1
   Title: Tractor falls 10 feet off ghat in Odisha, 12 injured in mishap
   URL: https://odishatv.in/odisha/tractor-falls-10-feet-off-ghat-in-odisha-12-injured-in-mishap-10514590
   Published: 2025-09-29 22:54:22
   Authors: Suranjan Mishra
   Image: Yes
   Image URL: https://img-cdn.publive.online/fit-in/1280x960/odishatv/media/media_files/2025/09/29/tractor-falls-10-feet-off-ghat-in-odisha-12-injured-in-mishap-2025-09-29-22-41-13.jpeg
   Content Length: 1829 characters
   Content Preview: A road accident near Dhepaguda under Adaba police limits in Gajapati district left 12 people injured on Monday after a tractor lost balanc

In [ ]:
# Debug image extraction for problematic sources
print("🔍 DEBUGGING IMAGE EXTRACTION")
print("=" * 50)

# Set logging to DEBUG level to see detailed image extraction logs
logging.getLogger().setLevel(logging.DEBUG)

# Debug Orissa Post
debug_image_extraction("orissapost", max_articles=2)

# Debug Odisha Bytes  
debug_image_extraction("odishabytes", max_articles=2)

# Reset logging level
logging.getLogger().setLevel(logging.INFO)
